<a href="https://colab.research.google.com/github/prasannakumar9i/ev-llm-diagnostic-assistant/blob/main/ev_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV LLM Diagnostic Assistant (RAG Project)

Author: Sunny  
Project: EV AI Diagnostic Assistant  
Tools: Python, LangChain, FAISS, HuggingFace

## Day 1 – Project Setup

Goal:
- Create GitHub repository
- Connect Google Colab
- Initialize project notebook

#Day 2 Install libraries

In [35]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers accelerate
!pip install pypdf

In [36]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [37]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully


In [38]:
sentence = "EV battery draining quickly"

embedding = model.encode(sentence)

print(len(embedding))

384


# Day 3 — Download EV Repair Manuals Dataset

In this step, we collect EV repair manuals in PDF format.
These manuals will act as the knowledge base for the AI diagnostic assistant.
Later we will extract text from these PDFs and convert them into embeddings.

In [ ]:
import os

os.makedirs("ev_manuals", exist_ok=True)

print("EV manuals dataset folder created successfully")

In [ ]:
import os
os.listdir()

# Day 4 — Load PDF Manuals and Extract Text

In this step we upload a PDF manual into Google Colab and use
LangChain's PyPDFLoader to read the document.

Each page of the PDF will be converted into a document object
that will later be split into chunks for the RAG system.

#install pdf Libraries

In [1]:
from google.colab import files
uploaded = files.upload()

Saving harrier-ev-onwers-manual.pdf to harrier-ev-onwers-manual.pdf


In [2]:
!pip install langchain
!pip install langchain-community
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is 

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")

documents = loader.load()

print("Total pages loaded:", len(documents))

Total pages loaded: 422


In [4]:
print(documents[0].page_content[:500])

Following items are provided 
with your vehicle: 
1. Owner’s Manual
2. Battery Warranty Card
(if applicable)
3. First Aid Kit
4. Advance Warning Triangle
5. Jack
6. Spare Fuses (Provided in
fuse box)
7. Tool Kit
CAR IDENTIFICATION RECORD 
OWNER’S NAME: _____________________________________________________ 
ADDRESS: __________________________________________________________ 
SELLING DEALER CODE: _______________________________________________ 
DATE OF DELIVERY: ___________________________________


# Day 5 — Split Documents into Text Chunks

In this step we split the extracted manual text into smaller chunks.
Chunking improves retrieval accuracy in RAG systems because the
language model searches smaller text segments instead of full documents.

In [40]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [41]:
print(chunks[0].page_content[:1000])


Following items are provided 
with your vehicle: 
1. Owner’s Manual
2. Battery Warranty Card
(if applicable)
3. First Aid Kit
4. Advance Warning Triangle
5. Jack
6. Spare Fuses (Provided in
fuse box)
7. Tool Kit
CAR IDENTIFICATION RECORD 
OWNER’S NAME: _____________________________________________________ 
ADDRESS: __________________________________________________________ 
SELLING DEALER CODE: _______________________________________________ 
DATE OF DELIVERY: ___________________________________________________ 
DATE OF REGISTRATION: _______________________________________________ 
REGISTRATION NO.: ___________________________________________________ 
CHASSIS NO.: ________________________________________________________ 
MOTOR NO.: _________________________________________________________ 
       : _________________________________________________________ 
TRANSAXLE NO.: ______________________________________________________


#Day 6 Generate Embeddings

In [ ]:
#packages
!pip install langchain
!pip install sentence-transformers
!pip install langchain-community
!pip install langchain-text-splitters

In [ ]:
# imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#loading the pdf
loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")
documents = loader.load()

In [ ]:
# splitting the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(len(docs))

In [ ]:
#creating embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
#generating vectors
texts = [doc.page_content for doc in docs]

vectors = embeddings.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))

#Day 7 creating vector database

In [18]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.8 MB/s eta 0:00:00


In [23]:
#importing faiss
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [24]:
#load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_178/1611559615.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [43]:
#create vector database
vectorstore = FAISS.from_documents(documents, embeddings)

In [44]:
#testing
query = "battery not charging"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")


CHARGING 
49 
4. Open the protective cap on Charging  
Gun. 
 
5. Open the protective cap on Charging 
Inlet. 
 
6. Before connecting the charging gun to 
vehicle charging socket, make sure 
the gun lock is released. 
CAUTION 
 If the Gun Lock is not released 
please don’t insert the Charging Gun 
forcefully into the socket. It may 
damage the Charging Socket. 
 Don’t use the electric connection  
with extension or power strip for the 
slow charging or AC charging of the 
vehicle, this will lead to heat up the 
cables and charging gun. Prolong 
charging in such condition may lead 
to melting of wire and charging gun. 
7. If the actuator is engaged and the gun 
is not getting inserted properly, please 
turn the ignition ON -OFF via PEPS 
key. If issue still persists then contact 
TATA MO TORS EV Authorised ser-
vice center. 
8. Remove any dust on the Charging 
Gun and Charging Inlet. Connect the 
charging gun to vehicle AC Charging 
Inlet.
------
CHARGING 
56 
2. AC Charging (Wall Mou

In [45]:
vectorstore.save_local("ev_manual_vector_db")

In [46]:
query = "battery overheating"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

MAINTENANCE AND CARE  
378  
 
 NOTE 
Use only authorized Battery recom-
mended by TATA MOTORS. Use of 
any other unauthorized Battery will 
result into Intelligent Alternator Con-
trol (IAC) function deterioration. 
 
 
 
 
 
  NOTE 
 During normal operation, the bat-
tery generates gas which is explo-
sive in nature. A spark or open 
flame can cause the battery to ex-
plode causing very serious inju-
ries.  
 Keep all sparks, open flames and 
smoking materials away from the 
battery. 
 The battery contains sulphuric 
acid (electrolyte) which is poison-
ous and highly corrosive in nature. 
Getting electrolyte in your eyes or 
on the skin can cause severe 
burns. Wear protective clothing 
and a face shield or have a skilled 
technician to do the battery 
maintenance. 
TYRES 
 
1 Under inflation 
Excessive 
side tread 
wear 
2 Correct tyre pres-
sure 
Uniform 
wear 
3 Over inflation 
Excessive 
center 
tread wear
------
BATTERY AND COMPONENTENT 
 39 
HIGH VOLTAGE BATTERY SYS-
TEM 
Te

In [47]:
query = "charging port problem"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

INSTRUMENT CLUSTER  
 107 
Cleaning of Charging Inlet 
Covering the charging gun and charging inlet by dust cap will ensure protection from water and dust. 
Precautions to be Taken While Cleaning the Charging Inlet 
 Keep the vehicle lid always closed 
 When the lid is open ensure that dust caps are in closed position 
 During normal charging, make sure that DC charging cap is closed 
 In case of any dust/mud/snow accumulation in the charging port and also on CCS2 especially actuator area, it can be 
cleaned with blowing air before charging. 
 Allow the water to drain completely through drain holes. Allow the charging port to dry completely. 
NOTE 
Water entering into the charging port will always be drained through the drain system. If water is stagnant in charging port area call 
TATA MOTORS EV Authorised Service Centre to rectify the issue.
------
CHARGING 
49 
4. Open the protective cap on Charging  
Gun. 
 
5. Open the protective cap on Charging 
Inlet. 
 
6. Before connectin

In [48]:
from google.colab import files
files.download("ev_manual_vector_db/index.faiss")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Day 8 — Similarity Search Engine

In this section we load the FAISS vector database created in Day 7
and perform semantic similarity search on EV repair manuals.

In [ ]:
#Libraries
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu

In [ ]:
#importing libraries
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
#loading embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
#loading FAISS vector Database
vectorstore = FAISS.load_local("ev_manual_vector_db",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
query = "why EV battery not charging"

In [ ]:
results = vectorstore.similarity_search(query, k=3)

In [ ]:
for r in results:
  print(r.page_content)
  print("------------")

In [ ]:
query = "charging port problem"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------------")

#Day 9 Integrate LLM

In [6]:
!pip install transformers accelerate
from transformers import pipeline

In [10]:
#Load an LLM
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=256
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

In [11]:
#creating a prompt template
def generate_answer(context, question):

    prompt = f"""
    You are an EV diagnostic assistant.

    Use the following EV repair manual information to answer the question.

    Manual Information:
    {context}

    Question:
    {question}

    Provide a clear diagnostic explanation.
    """

    result = llm(prompt)

    return result[0]["generated_text"]

In [52]:
query = "Why is my EV battery overheating?"

documents = vectorstore.similarity_search(query, k=3)

context = " ".join([doc.page_content for doc in documents])

answer = generate_answer(context, query)

print(answer)

Token indices sequence length is longer than the specified maximum sequence length for this model (1446 > 512). Running this sequence through the model will result in indexing errors



    You are an EV diagnostic assistant.

    Use the following EV repair manual information to answer the question.

    Manual Information:
    BATTERY AND COMPONENTENT 
 39 
HIGH VOLTAGE BATTERY SYS-
TEM 
Temperature Limits  
Battery pack and vehicle can  operate 
safely in limits from 0°C to 45°C.  
NOTE 
To control the battery temperature of 
the high voltage battery, the chiller is 
used to cool down the battery It may 
switch on automatically without re-
quest from control panel which may 
generate noise from operation of the 
air conditioner compressor and cool-
ing fan. 
HV Battery Life & Maintenance 
This Vehicle comes with a standard bat-
tery warranty as mentioned in warranty 
section. Regular service of the vehicle 
and charging protocol to be followed to 
maximize the battery life. 
Energy Information 
The vehicle battery pack has a maximum 
energy as specified in Tec hnical Specifi-
cation. Energy retention capacity deterio-
rates over several cycles of usage and 
hence 

#day 10 add source citations from from ev manual

In [53]:
query = "Why is my EV battery overheating?"

documents = vectorstore.similarity_search(query, k=3)

context = " ".join([doc.page_content for doc in documents])

sources = [doc.metadata["page"] for doc in documents]

In [54]:
answer = generate_answer(context, query)

print("Answer:\n")
print(answer)

Answer:


    You are an EV diagnostic assistant.

    Use the following EV repair manual information to answer the question.

    Manual Information:
    BATTERY AND COMPONENTENT 
 39 
HIGH VOLTAGE BATTERY SYS-
TEM 
Temperature Limits  
Battery pack and vehicle can  operate 
safely in limits from 0°C to 45°C.  
NOTE 
To control the battery temperature of 
the high voltage battery, the chiller is 
used to cool down the battery It may 
switch on automatically without re-
quest from control panel which may 
generate noise from operation of the 
air conditioner compressor and cool-
ing fan. 
HV Battery Life & Maintenance 
This Vehicle comes with a standard bat-
tery warranty as mentioned in warranty 
section. Regular service of the vehicle 
and charging protocol to be followed to 
maximize the battery life. 
Energy Information 
The vehicle battery pack has a maximum 
energy as specified in Tec hnical Specifi-
cation. Energy retention capacity deterio-
rates over several cycles of usage an

In [55]:
print("\nSources from EV Manual:")

for s in sources:
    print("Page:", s)


Sources from EV Manual:
Page: 47
Page: 10
Page: 49


In [56]:
query = "Why is my EV battery overheating?"

documents = vectorstore.similarity_search(query, k=3)

context = " ".join([doc.page_content for doc in documents])

sources = [doc.metadata["page"] for doc in documents]

answer = generate_answer(context, query)

print("Answer:\n")
print(answer)

print("\nSources from EV Manual:")

for s in sources:
    print("Page:", s)

Answer:


    You are an EV diagnostic assistant.

    Use the following EV repair manual information to answer the question.

    Manual Information:
    BATTERY AND COMPONENTENT 
 39 
HIGH VOLTAGE BATTERY SYS-
TEM 
Temperature Limits  
Battery pack and vehicle can  operate 
safely in limits from 0°C to 45°C.  
NOTE 
To control the battery temperature of 
the high voltage battery, the chiller is 
used to cool down the battery It may 
switch on automatically without re-
quest from control panel which may 
generate noise from operation of the 
air conditioner compressor and cool-
ing fan. 
HV Battery Life & Maintenance 
This Vehicle comes with a standard bat-
tery warranty as mentioned in warranty 
section. Regular service of the vehicle 
and charging protocol to be followed to 
maximize the battery life. 
Energy Information 
The vehicle battery pack has a maximum 
energy as specified in Tec hnical Specifi-
cation. Energy retention capacity deterio-
rates over several cycles of usage an